# Setup

In [2]:
# Fabric notebook parameters. The pipeline's ForEach passes one source per
# iteration, so a failure is isolated to that source; None means "every enabled
# source", which is what a manual run wants.
#
# This cell must stay tagged `parameters` — Fabric injects the pipeline's values
# in a new cell directly below it, so anything defined here is a default, not a
# constant.
source_name = None

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 9, Finished, Available, Finished, False)

In [3]:
# Environment bootstrap. A Fabric notebook has neither the repo root on
# sys.path nor as its working directory, so relative paths like
# "config/config.yaml" cannot resolve there. Detecting the OneLake mount keeps
# one notebook working in both places instead of maintaining two copies.
import os
import sys

CODE_ROOT = "/lakehouse/default/Files/code"
IN_FABRIC = os.path.isdir(CODE_ROOT)

if IN_FABRIC and CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

CONFIG_DIR = f"{CODE_ROOT}/config" if IN_FABRIC else "config"

# In Fabric, secrets come from the workspace rather than a gitignored .env.
# Set them here for a trial run; Chapter 8 replaces this with Key Vault.
if IN_FABRIC:
    os.environ.setdefault("ADZUNA_APP_ID", "<your-adzuna-app-id>")
    os.environ.setdefault("ADZUNA_APP_KEY", "<your-adzuna-app-key>")
    os.environ.setdefault("JOOBLE_API_KEY", "<your-jooble-api-key>")
    os.environ.setdefault("APP_ENV", "fabric")

print("Running in Fabric" if IN_FABRIC else "Running locally")

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 10, Finished, Available, Finished, False)

Running in Fabric


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from transformation.bronze_writer import run_bronze_ingestion
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger

setup_logging(config_path=f"{CONFIG_DIR}/logging_config.yaml")
logger = get_logger(__name__)

# Fabric provides a Delta-enabled session already; getOrCreate() returns it.
spark = SparkSession.builder.appName("bronze_ingestion").getOrCreate()

app_config = load_config(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
)

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 11, Finished, Available, Finished, False)

# Paths

In [5]:
# The same OneLake location needs two different path forms, depending on which
# library does the reading. Plain Python IO (run_ingestion's json.dump,
# bronze_writer's Path.glob) needs the local mount; Spark takes the path
# relative to the notebook's default lakehouse. Mixing them up produces an
# empty DataFrame rather than an error, so it is worth being explicit.
BRONZE_JSON_DIR = "/lakehouse/default/Files/bronze" if IN_FABRIC else "data/bronze"
BRONZE_TABLE_PATH = "Tables/bronze_job_postings" if IN_FABRIC else "data/delta/bronze_job_postings"

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 12, Finished, Available, Finished, False)

# Run ingestion (Chapter 2's connectors), then write Bronze

`run()` takes every path as an argument rather than loading config off a
hardcoded relative path — that is what lets this same call work from a
Fabric notebook, a local CLI run, and CI without any of them needing to
chdir or edit files on disk.

In [6]:
from ingestion.run_ingestion import run as run_connectors

exit_code = run_connectors(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
    logging_config_path=f"{CONFIG_DIR}/logging_config.yaml",
    bronze_path=BRONZE_JSON_DIR,
    source_name=source_name,
)

# 0 = every source succeeded; 1 = at least one source failed (see logs above).
assert exit_code == 0, f"Ingestion reported failures, exit_code={exit_code}"


StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 13, Finished, Available, Finished, False)

2026-08-28 15:39:09 | INFO     | run_id=766ee6a6 | ingestion.run_ingestion | Starting ingestion run_id=766ee6a6
2026-08-28 15:39:11 | INFO     | run_id=766ee6a6 | ingestion.base_connector | [adzuna] country=us page=1 fetched=50 records (running total=50)
2026-08-28 15:39:12 | INFO     | run_id=766ee6a6 | ingestion.base_connector | [adzuna] country=us page=2 fetched=50 records (running total=100)
2026-08-28 15:39:13 | INFO     | run_id=766ee6a6 | ingestion.base_connector | [adzuna] country=us page=3 fetched=50 records (running total=150)
2026-08-28 15:39:16 | INFO     | run_id=766ee6a6 | ingestion.base_connector | [adzuna] country=us page=4 fetched=50 records (running total=200)
2026-08-28 15:39:18 | INFO     | run_id=766ee6a6 | ingestion.base_connector | [adzuna] country=us page=5 fetched=50 records (running total=250)
2026-08-28 15:39:18 | INFO     | run_id=766ee6a6 | ingestion.run_ingestion | Wrote 250 records to /lakehouse/default/Files/bronze/adzuna_us_766ee6a6.json
2026-08-28 15:3

# Read the connector JSON output and write the Bronze Delta table

In [7]:
bronze_df = run_bronze_ingestion(
    spark,
    json_dir=BRONZE_JSON_DIR,
    table_path=BRONZE_TABLE_PATH,
)
bronze_df.show(5, truncate=50)

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 14, Finished, Available, Finished, False)

2026-08-28 15:39:31 | INFO     | run_id=766ee6a6 | transformation.bronze_writer | Loaded 1750 raw records from 9 files in /lakehouse/default/Files/bronze
2026-08-28 15:39:42 | INFO     | run_id=766ee6a6 | transformation.bronze_writer | Wrote 1750 rows to Bronze Delta table at Tables/bronze_job_postings across partitions: [datetime.date(2026, 8, 28), datetime.date(2026, 8, 20), datetime.date(2026, 8, 23)]
+------+--------------------+-----------------------+-------------------------------+-------------+-------+--------------------------------------------------+----------+----------+--------+------+---------------------------------+-------------------------------------------+--------------------------------+--------+--------------------------------------------------+--------------+
|source|       source_job_id|                  title|                        company| location_raw|country|                                       description|salary_min|salary_max|currency|remote|             

# Sanity checks before trusting this run

A row count of 0 here almost always means `BRONZE_JSON_DIR` is wrong, not
that the APIs returned nothing: `Path.glob` on a missing directory yields
nothing silently instead of raising.

In [8]:
print("Row count:", bronze_df.count())
print("Schema:")
bronze_df.printSchema()
print("Records per source:")
bronze_df.groupBy("source").count().show()

StatementMeta(, b242b94e-b745-4672-b501-d4b0709c7e18, 15, Finished, Available, Finished, False)

Row count: 1750
Schema:
root
 |-- source: string (nullable = false)
 |-- source_job_id: string (nullable = false)
 |-- title: string (nullable = true)
 |-- company: string (nullable = true)
 |-- location_raw: string (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- salary_min: double (nullable = true)
 |-- salary_max: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- remote: boolean (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- url: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- raw_payload: string (nullable = true)
 |-- ingestion_date: date (nullable = true)

Records per source:
+------+-----+
|source|count|
+------+-----+
|jooble|  750|
|adzuna| 1000|
+------+-----+

